# Exercise 73 — Mini-project: Incremental loader

## Problem

A source CSV is appended to over time. You want to keep a SQLite table in sync, **without re-loading everything every time**. The pattern:

1. Maintain a **watermark** — the highest `event_ts` already loaded.
2. On each run, read only rows from the source whose `event_ts > watermark`.
3. Insert them, update the watermark.

This is how most batch "CDC-lite" pipelines work — much simpler than streaming, and good enough for the majority of ETL.

## Schema

Source CSV:
```
id,event_ts,user,action
```

Sink SQLite table `events`:
```
id INTEGER PRIMARY KEY, event_ts TEXT, user TEXT, action TEXT
```

A tiny `watermarks` table tracks the latest loaded timestamp per source:
```
source TEXT PRIMARY KEY, last_ts TEXT
```

## Setup

In [ ]:
import sqlite3
from pathlib import Path

DB = "incremental.db"
CSV = "events.csv"
Path(DB).unlink(missing_ok=True)

# Initial source file with 3 rows
Path(CSV).write_text(
    "id,event_ts,user,action\n"
    "1,2026-01-01T10:00,alice,login\n"
    "2,2026-01-01T10:05,bob,login\n"
    "3,2026-01-01T10:10,alice,view\n",
    encoding="utf-8",
)
print("setup ok")


## Task 1 — Initialize sink and watermark tables

Write a function that creates the two tables if they don't exist.

```python
def setup_sink(db_path: str) -> None:
    """Create 'events' and 'watermarks' tables. Idempotent (safe to call multiple times)."""
```

Keep table creation idempotent with `CREATE TABLE IF NOT EXISTS`. Re-running the loader should not blow up.

In [ ]:
import sqlite3

def setup_sink(db_path: str) -> None:
    pass  # TODO

setup_sink(DB)
setup_sink(DB)   # idempotent

with sqlite3.connect(DB) as conn:
    tables = {r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
assert {"events", "watermarks"} <= tables
print("ok")


## Task 2 — Read the watermark

Write a function that returns the current watermark for a given source name. If no row exists, return a default (`"1970-01-01T00:00"`) — guarantees the first run loads everything.

```python
def get_watermark(db_path: str, source: str) -> str:
    ...
```

In [ ]:
import sqlite3

def get_watermark(db_path: str, source: str) -> str:
    pass  # TODO

# No watermark set yet → default
assert get_watermark(DB, "events") == "1970-01-01T00:00"
print("ok")


## Task 3 — Load only rows past the watermark

Write a function that:
1. Reads the watermark.
2. Streams the source CSV.
3. Inserts rows where `event_ts > watermark`.
4. Updates the watermark to the **max** `event_ts` seen (across new rows).
5. Returns the count of rows inserted.

```python
def load_new(db_path: str, csv_path: str, source: str) -> int:
    ...
```

Do the insert and watermark update in **one transaction**: either both succeed or neither does. Use `with conn:` (commits on clean exit, rolls back on exception).

Use `INSERT OR IGNORE` for the `events` insert — so re-running with overlapping ranges doesn't double-insert by primary key.

Watermark upsert in SQLite:
```sql
INSERT INTO watermarks(source, last_ts) VALUES (?, ?)
ON CONFLICT(source) DO UPDATE SET last_ts = excluded.last_ts
```

In [ ]:
import csv
import sqlite3

def load_new(db_path: str, csv_path: str, source: str) -> int:
    pass  # TODO

# First run — loads everything (3 rows)
n = load_new(DB, CSV, "events")
assert n == 3
with sqlite3.connect(DB) as conn:
    cnt = conn.execute("SELECT COUNT(*) FROM events").fetchone()[0]
assert cnt == 3
assert get_watermark(DB, "events") == "2026-01-01T10:10"

# Second run on the same CSV — nothing new
n2 = load_new(DB, CSV, "events")
assert n2 == 0
with sqlite3.connect(DB) as conn:
    cnt = conn.execute("SELECT COUNT(*) FROM events").fetchone()[0]
assert cnt == 3       # no duplicates
print("ok")


## Task 4 — Source grows; loader picks up new rows only

Append two new rows to the CSV. Run the loader again. Only those two rows should be inserted, and the watermark should advance.

This cell exercises the function from Task 3 — you don't need to write new code, just verify.

In [ ]:
import sqlite3
from pathlib import Path

with open(CSV, "a", encoding="utf-8") as f:
    f.write("4,2026-01-01T11:00,carol,login\n")
    f.write("5,2026-01-01T11:15,alice,logout\n")

n = load_new(DB, CSV, "events")
assert n == 2
with sqlite3.connect(DB) as conn:
    cnt = conn.execute("SELECT COUNT(*) FROM events").fetchone()[0]
assert cnt == 5
assert get_watermark(DB, "events") == "2026-01-01T11:15"

# Idempotent — running again does nothing
assert load_new(DB, CSV, "events") == 0
print("ok")
